# Comparing Docker Compose, Swarm, and Kubernetes for Local Orchestration

## Purpose

Compose, Swarm, and Kubernetes all orchestrate the same containers, but they answer different questions. Compose answers "how do these containers run together on one machine"; Swarm answers "how do they run spread across several Docker nodes"; Kubernetes answers "how do they run on a general-purpose cluster with its own API objects". This notebook scaffolds the same demo app (a web front end plus a cache) in all three formats and compares them structurally, so the trade-offs are visible side by side before committing to one for local work.

## When to use

- Reach for **Compose** for single-host development and integration-test environments: one file, one command, no cluster to install.
- Reach for **Swarm** when the next step after Compose is a small multi-node Docker setup and the team wants to keep the Compose file format.
- Reach for **Kubernetes** manifests when the destination is a shared or hosted cluster and the workload needs the wider ecosystem (ingress controllers, autoscalers, GitOps operators).

## Prerequisites

- A container runtime on the workstation for actually running things (`docker compose up` needs only the engine; `docker stack deploy` additionally needs Swarm mode initialised; `kubectl apply` needs a reachable cluster such as a local one).
- Nothing extra for this notebook itself: every code cell below uses only the Python standard library and writes its files to a temporary directory.

In [ ]:
# last_verified: 2026-09-20 · Docker n/a
"""Scaffold the same demo app (web + cache) for three orchestrators."""
import tempfile
from pathlib import Path

COMPOSE_YAML = """services:
  web:
    image: nginx:alpine
    ports:
      - "8080:80"
    depends_on:
      - cache
    deploy:
      resources:
        limits:
          cpus: "0.50"
          memory: 128M
  cache:
    image: redis:alpine
"""

STACK_YAML = """services:
  web:
    image: nginx:alpine
    ports:
      - "8080:80"
    depends_on:
      - cache
    deploy:
      replicas: 2
      update_config:
        parallelism: 1
        order: start-first
      restart_policy:
        condition: on-failure
      resources:
        limits:
          cpus: "0.50"
          memory: 128M
  cache:
    image: redis:alpine
    deploy:
      replicas: 1
"""

K8S_YAML = """apiVersion: apps/v1
kind: Deployment
metadata:
  name: web
spec:
  replicas: 2
  selector:
    matchLabels:
      app: web
  template:
    metadata:
      labels:
        app: web
    spec:
      containers:
        - name: web
          image: nginx:alpine
          ports:
            - containerPort: 80
---
apiVersion: v1
kind: Service
metadata:
  name: web
spec:
  selector:
    app: web
  ports:
    - port: 80
      targetPort: 80
---
apiVersion: apps/v1
kind: Deployment
metadata:
  name: cache
spec:
  replicas: 1
  selector:
    matchLabels:
      app: cache
  template:
    metadata:
      labels:
        app: cache
    spec:
      containers:
        - name: cache
          image: redis:alpine
"""

WORKDIR = Path(tempfile.mkdtemp(prefix="orch-compare-"))
FILES = {
    "compose.yaml": COMPOSE_YAML,
    "stack.yaml": STACK_YAML,
    "k8s.yaml": K8S_YAML,
}
for name, body in FILES.items():
    (WORKDIR / name).write_text(body)
    print(f"wrote {WORKDIR / name} ({len(body.splitlines())} lines)")

In [ ]:
"""Compare the three scaffolds structurally and print a decision table."""
texts = {name: (WORKDIR / name).read_text() for name in FILES}

rows = [
    ("Scope", "single host", "multi-node Swarm", "any cluster"),
    ("Replica control", "restart only (no replicas key)", "deploy.replicas: 2", "spec.replicas: 2"),
    (
        "Rolling update",
        "recreate via up --force-recreate",
        "deploy.update_config",
        "built-in rollout (kubectl rollout status)",
    ),
    ("Services defined", str(texts["compose.yaml"].count("image:")), str(texts["stack.yaml"].count("image:")), str(texts["k8s.yaml"].count("image:")),),
    ("Local run command", "docker compose up", "docker stack deploy -c stack.yaml demo", "kubectl apply -f k8s.yaml"),
]
width = max(len(r[0]) for r in rows)
print(f"{'Aspect':<{width}} | Compose | Swarm stack | Kubernetes")
print("-" * (width + 55))
for aspect, compose, swarm, k8s in rows:
    print(f"{aspect:<{width}} | {compose} | {swarm} | {k8s}")

## Reading the comparison

- **Compose is the shortest path locally.** No scheduler, no cluster state: the file declares containers, networks, and volumes, and `docker compose up` converges the single host to match. Scaling past one machine is where it stops.
- **Swarm reuses the Compose format and adds scheduling.** The `stack.yaml` above is nearly identical to `compose.yaml` plus a `deploy` block (replicas, update policy, restart policy). The cost is a Swarm to join: the manager node must be initialised before the first `docker stack deploy`, and placement is still Docker-centric.
- **Kubernetes replaces the file format with API objects.** Deployments carry replicas and rollout behaviour natively, Services decouple discovery from pods, and `kubectl apply -f` works against any conformant cluster. The cost is conceptual surface: even this tiny app needs three documents and label selectors that must agree.

Rule of thumb for local work: prototype on Compose; move to Swarm only if multi-node Docker is genuinely the target; write Kubernetes manifests once a cluster (local or remote) is part of the workflow and the app needs rollout history, self-healing beyond restart policies, or cluster-wide add-ons.

In [ ]:
"""Verify the scaffolds: files exist and carry the expected structure."""
checks = [
    ("compose.yaml exists", (WORKDIR / "compose.yaml").exists()),
    ("stack.yaml exists", (WORKDIR / "stack.yaml").exists()),
    ("k8s.yaml exists", (WORKDIR / "k8s.yaml").exists()),
    ("compose declares 2 images", (WORKDIR / "compose.yaml").read_text().count("image:") == 2),
    ("stack sets web replicas: 2", "replicas: 2" in (WORKDIR / "stack.yaml").read_text()),
    ("stack has update_config", "update_config:" in (WORKDIR / "stack.yaml").read_text()),
    ("k8s has Deployment + Service", (WORKDIR / "k8s.yaml").read_text().count("kind:") == 3),
    ("k8s Service targets port 80", "targetPort: 80" in (WORKDIR / "k8s.yaml").read_text()),
    ("no obsolete version: key in compose", not (WORKDIR / "compose.yaml").read_text().startswith("version:")),
]
for label, ok in checks:
    print(f"{'PASS' if ok else 'FAIL'}: {label}")
assert all(ok for _, ok in checks), "verify failed"

## Common errors

- `docker stack deploy` on a node that never joined a Swarm fails with a "not a swarm manager" error. Fix: initialise or join first (`docker swarm init` on a fresh single node), then redeploy.
- Typing `docker-compose` when only the v2 plugin is installed (or vice versa) gives "command not found". Fix: use `docker compose` (space, plugin) on current installs; keep the hyphen form only where the standalone binary is still present.
- A top-level `version:` key in a Compose file triggers an "obsolete" warning and is ignored. Fix: delete the key; the Compose Spec no longer uses it.
- `kubectl apply` seemingly succeeding but nothing changing in the local app usually means the wrong context is active. Fix: confirm with `kubectl config current-context` before applying.
- Label selectors that disagree (`spec.selector.matchLabels` vs pod template `labels`) make a Deployment's pods invisible to its own rollout and to Services. Fix: keep the `app: <name>` value identical in all three places, as in the scaffold above.

## References

- [Production Compose stack](../manifests/production-compose-stack.yaml) — the multi-service Compose manifest already in this kit; compare its healthchecks and resource limits with the minimal scaffold above.
- [Comparing Docker networking drivers](comparing-docker-networking-drivers.ipynb) — the companion notebook on bridge/host/overlay/macvlan; useful background once the scaffolded services need custom networks.

## What to try next

Add a named volume for the cache to the Compose file, mirror it as a Swarm `volumes` entry and a Kubernetes PersistentVolumeClaim, and re-run the verify cell to confirm all three still converge. A follow-up drill could practise `kubectl rollout restart` against the local cluster and watch the Service endpoints stay available.